[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [21]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [22]:
import torch
import torch.nn as nn
import math

In [23]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.num_heads = num_heads
        assert(d_model % self.num_heads == 0)
        self.head_dim = d_model // self.num_heads

    def forward(self, x_q, x_kv):
        B, Sq, _ = x_q.size()
        _, Skv, _ = x_kv.size()

        q = self.W_q(x_q) # [B, S_q, D] -> [B, S_q, D]
        k = self.W_k(x_kv) # [B, S_kv, D] -> [B, S_kv, D]
        v = self.W_v(x_kv) # [B, S_kv, D] -> [B, S_kv, D]

        q_multi = q.view(B, Sq, self.num_heads, self.head_dim).transpose(1, 2) # [B, S_q, D] -> [B, H, S_q, Dh]
        k_multi = k.view(B, Skv, self.num_heads, self.head_dim).transpose(1, 2) # [B, S_kv, D] -> [B, H, S_kv, Dh]
        v_multi = v.view(B, Skv, self.num_heads, self.head_dim).transpose(1, 2) # [B, S_kv, D] -> [B, H, S_kv, Dh]

        attn_score_intermediate = q_multi @ k_multi.mT # [B, H, S_q, Dh] @ [B, H, Dh, S_kv] -> [B, H, S_q, S_kv]
        attn_score = torch.softmax(attn_score_intermediate / math.sqrt(self.head_dim), -1) # [B, H, S_q, S_kv]
        attn = attn_score @ v_multi # [B, H, S_q, S_kv] @ [B, H, S_kv, Dh] -> [B, H, S_q, Dh]
        attn = attn.transpose(1, 2).contiguous() # [B, S_q, H, Dh]
        attn = attn.view(B, Sq, self.head_dim * self.num_heads) # [B, S_q, D]
        return self.W_o(attn) # [B, S_q, D] [D, D] -> [B, S_q, D]


In [24]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [25]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.6ms)
  ✅ [2/4] Q and KV different lengths (1.1ms)
  ✅ [3/4] No causal mask — all KV affects all Q (3.2ms)
  ✅ [4/4] Gradient flow (1.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.0ms total)
  Progress saved. Run status() to see your dashboard.

